# Talk to the People — Colab Setup

Run this notebook in **Google Colab** (Runtime → Change runtime type → GPU) or in **Cursor** with a Python kernel.

Uses **Hugging Face Qwen2.5-3B-Instruct** for local inference. Enable GPU for faster embeddings and generation.

## 1. Check environment

In [1]:
import sys
IN_COLAB = "google.colab" in sys.modules
print(f"Running in Colab: {IN_COLAB}")
try:
    import torch
    print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
except ImportError:
    print("PyTorch not yet installed")

!nvidia-smi

Running in Colab: True
PyTorch CUDA available: True
Mon Feb 23 14:50:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/

## 2. Mount Google Drive (Colab only)

Mount Drive to persist eval results. Results will be saved under `My Drive/Talk-to-the-People/eval/results/`.

In [2]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    EVAL_OUTPUT_DIR = "/content/drive/MyDrive/Talk-to-the-People/eval/results"
    import os
    os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)
    print(f"Eval results will be saved to: {EVAL_OUTPUT_DIR}")
else:
    EVAL_OUTPUT_DIR = "eval/results"

Mounted at /content/drive
Eval results will be saved to: /content/drive/MyDrive/Talk-to-the-People/eval/results


## 3. Install dependencies

In [3]:
# Colab: clone or pull latest, then install from requirements.txt
import os
REPO_DIR = "/content/Talk-to-the-People-A-Grounded-Persona-Conversation-System"
RE_CLONE = True  # Set True to remove and re-clone (for major updates)

if IN_COLAB:
    if os.path.exists(REPO_DIR):
        if RE_CLONE:
            !rm -rf {REPO_DIR}
            !git clone https://github.com/Jennt54321/Talk-to-the-People-A-Grounded-Persona-Conversation-System.git
        else:
            %cd {REPO_DIR}
            !git pull origin main
            %cd /content
    else:
        !git clone https://github.com/Jennt54321/Talk-to-the-People-A-Grounded-Persona-Conversation-System.git
    %pip install -q -r {REPO_DIR}/requirements.txt
else:
    %pip install -q -r requirements.txt

Cloning into 'Talk-to-the-People-A-Grounded-Persona-Conversation-System'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 128 (delta 43), reused 111 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 22.50 MiB | 16.25 MiB/s, done.
Resolving deltas: 100% (43/43), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 4. Project setup

**In Colab:** Project is cloned from GitHub. `%cd` into project root.

**In Cursor:** The notebook is already in the project; `PROJECT_DIR` will auto-detect.

In [4]:
from pathlib import Path

PROJECT_DIR = Path("/content/Talk-to-the-People-A-Grounded-Persona-Conversation-System")

if IN_COLAB:
    if not PROJECT_DIR.exists():
        !git clone https://github.com/Jennt54321/Talk-to-the-People-A-Grounded-Persona-Conversation-System.git
    get_ipython().run_line_magic('cd', str(PROJECT_DIR))
else:
    # Cursor / local: use cwd (run notebook from project root)
    PROJECT_DIR = Path.cwd()
    assert (PROJECT_DIR / "app" / "main.py").exists(), f"Project root not found. cd to project root first."

print(f"Project root: {PROJECT_DIR}")
sys.path.insert(0, str(PROJECT_DIR))

/content/Talk-to-the-People-A-Grounded-Persona-Conversation-System
Project root: /content/Talk-to-the-People-A-Grounded-Persona-Conversation-System


## 7. Run Eval

Execute the evaluation pipeline (retrieval -> generation -> A/B/C metrics).  
Results are saved to Google Drive when mounted (see step 2).

**To continue from existing `eval_retrieval.json`:** Set `CONTINUE_FROM_RETRIEVAL = True` below to skip Stage 1 (retrieval) and resume from generation.

**Speed tips:** Set `EVAL_LIMIT` (e.g. `10`) to run fewer questions; reduce `EVAL_TOP_K` (e.g. `4`) for fewer chunks per question; use `--no-rerank` for faster retrieval (lower quality).

In [5]:
import os
from pathlib import Path
os.chdir(PROJECT_DIR)

# Set True to skip retrieval and continue from existing eval_retrieval.json
CONTINUE_FROM_RETRIEVAL = True
retrieval_path = Path(EVAL_OUTPUT_DIR) / "eval_retrieval.json"

if CONTINUE_FROM_RETRIEVAL and retrieval_path.exists():
    print(f"Continuing from {retrieval_path}")
    !python eval/run_eval.py -q eval/questions_life.json -o "{EVAL_OUTPUT_DIR}" --from-retrieval "{retrieval_path}"
else:
    !python eval/run_eval.py -q eval/questions_life.json -o "{EVAL_OUTPUT_DIR}"

Continuing from /content/drive/MyDrive/Talk-to-the-People/eval/results/eval_retrieval.json
Loading retrieval from /content/drive/MyDrive/Talk-to-the-People/eval/results/eval_retrieval.json (100 questions)...
modules.json: 100% 349/349 [00:00<00:00, 1.87MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 597kB/s]
README.md: 94.6kB [00:00, 75.7MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 252kB/s]
config.json: 100% 777/777 [00:00<00:00, 5.62MB/s]
model.safetensors: 100% 438M/438M [00:04<00:00, 107MB/s]  
Loading weights: 100% 199/199 [00:00<00:00, 1076.28it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100% 366/

## 8. Start the web app

In [5]:
import os
import subprocess
os.chdir(PROJECT_DIR)

if IN_COLAB:
    # Colab: run in background so you can run ngrok in the next cell
    subprocess.Popen(
        ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd=str(PROJECT_DIR),
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    import time
    time.sleep(3)
    print("App starting. Run the ngrok cell below to get a public URL.")
else:
    # Cursor / local: run in foreground (interrupt to stop)
    !uvicorn app.main:app --reload --port 8000

App starting. Run the ngrok cell below to get a public URL.


## 9. (Colab only) Expose with ngrok

Run in a **new cell** after the app has started to get a public URL:

In [7]:
if IN_COLAB:
    !pip install -q pyngrok
    from pyngrok import ngrok
    
    # Replace with your token from https://dashboard.ngrok.com/get-started/your-authtoken
    ngrok.set_auth_token("3A4hBkuBhaAG8cLaGPfzSC1irYq_7d5gDmimjZ3j2ho2qbXL7")
    
    public_url = ngrok.connect(8000)
    print(f"Open in browser: {public_url}")
else:
    print("Use http://localhost:8000")

Open in browser: NgrokTunnel: "https://dallyingly-laziest-alejandro.ngrok-free.dev" -> "http://localhost:8000"


In [8]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

True
Tesla T4
